
# NaviGator Toolkit: Demo of extracting file content and sending as query context


This notebook demonstrates extracting text from PDF or Word documents or using an image to generate a summary.

Many models and the LiteLLM API endpoint don't work well with uploaded files, so we extract the content of a file and send that as context in the query. This works with most models.

Note that you need to use an image capable model for image summarization.

### API key setup

You will need a NaviGator API key. Store the key in a `.json` file with the following format:

    {
      "OPENAI_API_KEY" : "Put your key here in the quotes",
      "base_url" : "https://api.ai.it.ufl.edu/"
    }

We recommend storing the file in your home directory to reduce the chance of accidentally committing it to a git repo. Anyone with your API key can use NaviGator as you! 

In [42]:
import openai
import os
import json
import base64
import mimetypes
import pypdf
import docx


## Load `.json` file with your key and API endpoint URL

In [ ]:
# Set the path to your json key file
key_file = '/home/magitz/navigator_api_keys.json'


# Load the JSON file
with open(key_file, 'r') as file:
    data = json.load(file)

# Extract the values
OPENAI_API_KEY = data.get('OPENAI_API_KEY')
base_url = data.get('base_url')

# Set the environment variable
os.environ['TOOLKIT_API_KEY'] = OPENAI_API_KEY


## Test connectivity and get model list

The reply should list the models available with your API key. An example (truncated) output is:

    SyncPage[Model](data=[Model(id='llama-3.1-70b-instruct', created=1677610602, object='model', owned_by='openai'), Model(id='sfr-embedding-mistral', created=1677610602, object='model', owned_by='openai'),...)], object='list')

In [44]:
# Check list of available models
client = openai.OpenAI(
    api_key=os.environ.get("TOOLKIT_API_KEY"),
    base_url=base_url
)

if not hasattr(client, "responses"):
    raise RuntimeError(
        "OpenAI SDK is missing the responses API. "
        "Upgrade to a newer openai package (e.g., pip install -U openai) "
        "or use a kernel/environment that includes responses."
    )

response = client.models.list()
 
print(response)

SyncPage[Model](data=[Model(id='llama-3.1-70b-instruct', created=1677610602, object='model', owned_by='openai'), Model(id='llama-3.1-8b-instruct', created=1677610602, object='model', owned_by='openai'), Model(id='llama-3.1-nemotron-nano-8B-v1', created=1677610602, object='model', owned_by='openai'), Model(id='llama-3.3-70b-instruct', created=1677610602, object='model', owned_by='openai'), Model(id='mistral-7b-instruct', created=1677610602, object='model', owned_by='openai'), Model(id='mistral-small-3.1', created=1677610602, object='model', owned_by='openai'), Model(id='nemotron-3-nano-30b-a3b', created=1677610602, object='model', owned_by='openai'), Model(id='nemotron-3-super-120b-a12b', created=1677610602, object='model', owned_by='openai'), Model(id='codestral-22b', created=1677610602, object='model', owned_by='openai'), Model(id='gemma-3-27b-it', created=1677610602, object='model', owned_by='openai'), Model(id='gpt-oss-20b', created=1677610602, object='model', owned_by='openai'), Mo

In [45]:
# Print available models in better format
for model in response:
    print(model.id)

llama-3.1-70b-instruct
llama-3.1-8b-instruct
llama-3.1-nemotron-nano-8B-v1
llama-3.3-70b-instruct
mistral-7b-instruct
mistral-small-3.1
nemotron-3-nano-30b-a3b
nemotron-3-super-120b-a12b
codestral-22b
gemma-3-27b-it
gpt-oss-20b
gpt-oss-120b
granite-3.3-8b-instruct
sfr-embedding-mistral
nomic-embed-text-v1.5
flux.1-dev
flux.1-schnell
whisper-large-v3
kokoro
o1
o3
o3-mini
o3-mini-high
o3-mini-medium
o4-mini
o4-mini-high
o4-mini-medium
gpt-4.1
gpt-4.1-mini
gpt-4.1-nano
gpt-4o
gpt-4o-mini
gpt-5
gpt-5-mini
gpt-5-nano
gpt-5.1
gpt-5.1-codex
gpt-5.1-codex-mini
gpt-5.2
gpt-5.2-codex
gpt-5.3-codex
gpt-5.4
gemini-2.0-flash
gemini-2.5-flash
gemini-2.5-pro
gemini-3.0-pro
gemini-3-pro
gemini-3.1-pro
claude-3-haiku
claude-3.5-haiku
claude-3-opus
claude-4.5-opus
claude-4.5-opus-thinking
claude-opus-4.6
claude-opus-4.6-thinking
claude-4.6-opus
claude-4.6-opus-thinking
claude-3.7-sonnet
claude-3.7-sonnet-thinking
claude-4-sonnet
claude-4.5-sonnet
claude-4.5-sonnet-thinking
claude-sonnet-4.6
claude-sonne

## Load a file

The `load_file()` function supports the following file types:

| Example File | Type | Extensions |
|------|-----------|--------|
| `data/Kossaifi_et_el_2026_NVIDIA_Medium-Range_Forecast.pdf` | PDF | `.pdf` |
| Not provided | Plain text | `.txt`, `.csv`, `.md`, etc. |
| `data/nemotron_info.docx` | Word document | `.docx` |
| `data/gator_swamp.png` |Image | `.png`, `.jpg`, `.jpeg`, `.gif`, `.webp` |




Change the `file` variable to point to any supported file.



In [55]:
def load_file(path):
    """
    Load a file and return its content for use as LLM context.

    Returns a dict with:
      - 'type': 'text' or 'image'
      - 'content': extracted text string (for text types)
      - 'image_data': base64-encoded data URI (for image types)
      - 'mime_type': detected MIME type
    """
    mime_type, _ = mimetypes.guess_type(path)
    ext = os.path.splitext(path)[1].lower()

    # --- PDF ---
    if ext == ".pdf" or mime_type == "application/pdf":
        reader = pypdf.PdfReader(path)
        text = "\n".join(page.extract_text() for page in reader.pages if page.extract_text())
        print(f"PDF extracted: {len(text)} characters")
        return {"type": "text", "content": text, "mime_type": mime_type}

    # --- Plain text ---
    elif mime_type and mime_type.startswith("text/"):
        with open(path, "r", encoding="utf-8", errors="replace") as f:
            text = f.read()
        print(f"Text file read: {len(text)} characters")
        return {"type": "text", "content": text, "mime_type": mime_type}

    # --- Word document (.docx) ---
    elif ext == ".docx" or mime_type == "application/vnd.openxmlformats-officedocument.wordprocessingml.document":
        doc = docx.Document(path)
        text = "\n".join(p.text for p in doc.paragraphs if p.text.strip())
        print(f"Word document extracted: {len(text)} characters")
        return {"type": "text", "content": text, "mime_type": mime_type}

    # --- Images ---
    elif mime_type and mime_type.startswith("image/"):
        with open(path, "rb") as f:
            data = base64.b64encode(f.read()).decode()
        image_data = f"data:{mime_type};base64,{data}"
        print(f"Image loaded: {os.path.basename(path)} ({mime_type})")
        print("Be sure your model supports image inputs to use this content effectively!")
        return {"type": "image", "image_data": image_data, "mime_type": mime_type}

    else:
        raise ValueError(f"Unsupported file type: {ext!r} (MIME: {mime_type})")


# --- Load your file here ---
# file = "data/Kossaifi_et_el_2026_NVIDIA_Medium-Range_Forecast.pdf"
# file = "data/nemotron_info.docx"
file = "data/gator_swamp.png"
file_info = load_file(file)


Image loaded: gator_swamp.png (image/png)
Be sure your model supports image inputs to use this content effectively!


## Create a summary of the file

In [56]:
# Define the prompt and model
if file_info["type"] == "text":
    prompt = "Summarize the key findings of the document."
    model = "nemotron-3-super-120b-a12b"
elif file_info["type"] == "image":
    prompt = "Describe the content of the image in detail, including any notable features or objects present."
    model = "gemma-3-27b-it"  # Must be a vision-capable model

try:
    print("Attempting to create a response...")

    if file_info["type"] == "text":
        # Text files: use the Responses API
        messages = [
            {
                "role": "system",
                "content": "You are a helpful assistant that provides a variety of AI file services."
            },
            {
                "role": "user",
                "content": f"Document content:\n\n{file_info['content']}\n\n{prompt}"
            }
        ]
        response = client.responses.create(
            model=model,
            input=messages
        )
        print(f"Response is:\n{response.output_text}\n")

    elif file_info["type"] == "image":
        # Images: use the Chat Completions API with image_url format
        messages = [
            {
                "role": "system",
                "content": "You are a helpful assistant that provides a variety of AI file services."
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {"url": file_info["image_data"]}
                    },
                    {
                        "type": "text",
                        "text": prompt
                    }
                ]
            }
        ]
        response = client.chat.completions.create(
            model=model,
            messages=messages
        )
        print(f"Response is:\n{response.choices[0].message.content}\n")

except Exception as e:
    print(f"An exception occurred while trying to create a chat session: {e}")


Attempting to create a response...
Response is:
Here's a detailed description of the image:

**Overall Impression:**

The image captures a breathtaking and dramatic scene of a swamp at sunset. The overall mood is serene yet slightly ominous, with a strong emphasis on the beauty of nature and the presence of wildlife.

**Key Elements:**

*   **Swamp Landscape:** The dominant feature is a dense swamp environment. Cypress trees, draped with Spanish moss, fill the background and sides of the image. The trees are silhouetted against the vibrant sunset sky.
*   **Water:** The water surface is calm, reflecting the colors of the sunset and the trees. There are some ripples and reflections creating an interesting texture.
*   **Alligator:** In the foreground, an alligator is swimming in the water. Only its head and back are visible above the surface, with its eyes and teeth clearly defined. The alligator is a focal point, adding a sense of wildness and potential danger to the scene.
*   **Sunse